In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path
import shutil
import os

DRIVE_PROJECT = Path(
    "/content/drive/MyDrive/kuliahS3/Dissertation_Writing/03_Papers/syarat_lulus/code"
)

LOCAL_PROJECT = Path("/content/dissertation_code")

if not DRIVE_PROJECT.exists():
    raise FileNotFoundError(
        f"Project tidak ditemukan:\n{DRIVE_PROJECT}"
    )

if LOCAL_PROJECT.exists():
    shutil.rmtree(LOCAL_PROJECT)

shutil.copytree(DRIVE_PROJECT, LOCAL_PROJECT)

print("Project copied to:")
print(LOCAL_PROJECT)

Project copied to:
/content/dissertation_code


In [3]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: Colab tidak menggunakan GPU.")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


Buat wrapper khusus Colab

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import numpy as np


ROOT = Path("/content/dissertation_code")

CONFIG_DIR = (
    ROOT
    / "stages/stage2_baselines/configs/multiseed"
)

RUNS_DIR = ROOT / "runs/stage2"

SEEDS = [
    20260827,
    20260828,
    20260829,
    20260830,
    20260831,
]


# ============================================================
# MODEL CONFIGURATION
# ============================================================

BPR_CONFIG = {
    "embedding_dim": 64,
    "learning_rate": 0.02,
    "l2": 0.001,
    "batch_size": 4096,
    "max_epochs": 100,
    "eval_every": 10,
    "patience_evaluations": 3,
    "min_delta": 0.00001,
}


LIGHTGCN_CONFIG = {
    "embedding_dim": 64,
    "layers": 1,
    "learning_rate": 0.005,
    "l2": 0.0001,
    "batch_size": 65536,
    "steps_per_epoch": 5,
    "max_epochs": 100,
    "eval_every": 10,
    "patience_evaluations": 4,
    "min_delta": 0.00001,

    # Dipertahankan dari konfigurasi lama.
    "cpu_threads": 4,
}


def create_config(model, seed):

    if model == "bpr_mf":
        base = BPR_CONFIG
        prefix = "BPRMF_MS"

    elif model == "lightgcn":
        base = LIGHTGCN_CONFIG
        prefix = "LIGHTGCN_MS"

    else:
        raise ValueError(f"Unknown model: {model}")

    config = {
        "run_id": f"{prefix}_{seed}",
        "seed": seed,
        **base,
    }

    CONFIG_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    config_path = (
        CONFIG_DIR
        / f"{prefix.lower()}_{seed}.json"
    )

    config_path.write_text(
        json.dumps(config, indent=2),
        encoding="utf-8",
    )

    return config_path


def run_model(model, seed):

    config_path = create_config(
        model,
        seed,
    )

    if model == "bpr_mf":
        runner = (
            ROOT
            / "stages/stage2_baselines/run_bpr_mf.py"
        )
    else:
        runner = (
            ROOT
            / "stages/stage2_baselines/run_lightgcn.py"
        )

    if not runner.exists():
        raise FileNotFoundError(
            f"Runner tidak ditemukan:\n{runner}"
        )

    print()
    print("=" * 70)
    print(f"MODEL : {model}")
    print(f"SEED  : {seed}")
    print(f"CONFIG: {config_path.name}")
    print("=" * 70)

    subprocess.run(
        [
            sys.executable,
            str(runner),
            "--config",
            str(config_path),
            "--quiet",
        ],
        cwd=ROOT,
        check=True,
    )

    prefix = (
        "BPRMF_MS"
        if model == "bpr_mf"
        else "LIGHTGCN_MS"
    )

    result_path = (
        ROOT
        / "runs/stage2"
        / model
        / f"{prefix}_{seed}"
        / "result.json"
    )

    if not result_path.exists():
        raise FileNotFoundError(
            "Training selesai tetapi result.json "
            f"tidak ditemukan:\n{result_path}"
        )

    return result_path


def summarize(rows):

    metrics = [
        "recall_at_10",
        "ndcg_at_10",
        "mrr_at_10",
    ]

    return {
        metric: {
            "mean": float(
                np.mean(
                    [r[metric] for r in rows]
                )
            ),
            "sd": float(
                np.std(
                    [r[metric] for r in rows],
                    ddof=1,
                )
            ),
        }
        for metric in metrics
    }


def main():

    report = {
        "environment": {
            "platform": "Google Colab",
            "gpu": "Tesla T4",
        },
        "seeds": SEEDS,
        "models": {},
    }

    for model in [
        "bpr_mf",
        "lightgcn",
    ]:

        rows = []

        for seed in SEEDS:

            result_path = run_model(
                model,
                seed,
            )

            result = json.loads(
                result_path.read_text(
                    encoding="utf-8"
                )
            )

            row = {
                "seed": seed,
                **result["metrics"],
                "best_epoch": result["best_epoch"],
                "runtime_seconds": result[
                    "runtime_seconds"
                ],
            }

            rows.append(row)

            print()
            print("RESULT")
            print(json.dumps(row, indent=2))

        report["models"][model] = {
            "runs": rows,
            "summary": summarize(rows),
        }

    output = (
        ROOT
        / "runs/stage2/selected_multiseed_summary.json"
    )

    output.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    output.write_text(
        json.dumps(
            report,
            indent=2,
        ),
        encoding="utf-8",
    )

    print()
    print("=" * 70)
    print("MULTISEED COMPLETED")
    print("=" * 70)
    print(output)

    return output


if __name__ == "__main__":
    main()